In [9]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

API_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"

# ======================
# Extracción
# ======================

@task
def extract_population(year_from: int = 2000, year_to: int = 2023, per_page: int = 1000):
    all_data = []
    page = 1

    while True:
        url = f"{API_URL}?format=json&date={year_from}:{year_to}&page={page}&per_page={per_page}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if len(data) < 2:
            break

        meta, rows = data
        all_data.extend(rows)

        print(f"🔄 Descargando página {page} de {meta['pages']} ({len(all_data)} filas acumuladas)")

        if page >= meta["pages"]:
            break
        page += 1

    return all_data

# ======================
# Transformación
# ======================

@task
def transform_population(data):
    df = pd.DataFrame(data)

    # Filtramos solo los válidos (iso3 no vacío)
    df = df[df["countryiso3code"].notna()]
    df = df[df["countryiso3code"].str.strip() != ""]

    # Columnas relevantes
    df = df[["countryiso3code", "country", "date", "value"]].copy()

    # Conversión de tipos
    df["year"] = df["date"].astype(int)
    df["population"] = df["value"].astype("Int64")  # admite nulos
    df = df.rename(columns={"countryiso3code": "id_country", "country": "country_name"})
    df = df[["id_country", "country_name", "year", "population"]]

    return df

# ======================
# Validación
# ======================

@task
def validate_population(df: pd.DataFrame) -> pd.DataFrame:
    schema = DataFrameSchema({
        "id_country": Column(str, [Check.str_length(3, 3)]),
        "country_name": Column(object, nullable=False),
        "year": Column(int, [Check.ge(1900), Check.le(2100)]),
        "population": Column(int, [Check.ge(0)], nullable=True),
    })
    return schema.validate(df)

# ======================
# Carga
# ======================

@task
def load_population(df: pd.DataFrame):
    file_path = Path("../stage/country_population.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"✅ Guardado {len(df)} filas en {file_path}")

# ======================
# Flow principal
# ======================

@flow(name="etl_population_total")
def etl_population_total(year_from: int = 2000, year_to: int = 2023):
    data = extract_population(year_from, year_to)
    df = transform_population(data)
    df = validate_population(df)
    load_population(df)

if __name__ == "__main__":
    etl_population_total()


05:31:35.419 | INFO    | Flow run 'turquoise-flamingo' - Beginning flow run 'turquoise-flamingo' for flow 'etl_population_total'

🔄 Descargando página 1 de 7 (1000 filas acumuladas)
🔄 Descargando página 2 de 7 (2000 filas acumuladas)
🔄 Descargando página 3 de 7 (3000 filas acumuladas)
🔄 Descargando página 4 de 7 (4000 filas acumuladas)
🔄 Descargando página 5 de 7 (5000 filas acumuladas)
🔄 Descargando página 6 de 7 (6000 filas acumuladas)
🔄 Descargando página 7 de 7 (6384 filas acumuladas)


05:32:07.213 | INFO    | Task run 'extract_population-cf2' - Finished in state Completed()

05:32:08.248 | INFO    | Task run 'transform_population-36e' - Finished in state Completed()

05:32:08.475 | INFO    | Task run 'validate_population-d99' - Finished in state Completed()

✅ Guardado 6264 filas en ..\stage\country_population.csv


05:32:08.707 | INFO    | Task run 'load_population-252' - Finished in state Completed()

05:32:08.718 | INFO    | Flow run 'turquoise-flamingo' - Finished in state Completed()